In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Number of Mutants per Project

In [ ]:
import pandas as pd

pd.read_sql_query("""
SELECT project_id, project_name, variant, sum(total) AS total
FROM mv_mutation_results_by_project_variant_mutator
WHERE variant IN ('ORIGINAL', 'INITIAL')
GROUP BY project_id, project_name, variant
ORDER BY project_id, variant_order(variant)
""", conn)

## Number of Mutants per Project + Mutator

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT project_id, project_name, variant, mutator, total
FROM mv_mutation_results_by_project_variant_mutator
WHERE variant = 'INITIAL'
""", conn)

df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def create_mutator_bar_chart(data):
    # Sort by project_id
    data = data.sort_values(['project_id'])

    # Create figure with adjusted size to accommodate the legend
    plt.figure(figsize=(18, 6))

    # Create combined project identifier (id + name)
    data['project_label'] = data['project_id'].astype(str) + ': ' + data['project_name']

    # Get unique project labels and mutators
    project_labels = data['project_label'].unique()
    mutators = sorted(data['mutator'].unique())  # Sort mutators for consistent indexing

    # Create mutator indices and labels for legend
    mutator_indices = {mutator: f"[{i+1}]" for i, mutator in enumerate(mutators)}
    mutator_legend_labels = [f"{idx} {mutator}" for mutator, idx in mutator_indices.items()]

    # Create a color map for mutators
    color_map = plt.colormaps['tab10']
    mutator_colors = {mutator: color_map(i % 10) for i, mutator in enumerate(mutators)}

    # Set up the plot
    ax = plt.subplot(111)
    bar_width = 0.8 / len(mutators)

    # For each project, plot bars for each mutator
    for i, project_label in enumerate(project_labels):
        project_data = data[data['project_label'] == project_label]

        # For each mutator, find its data for this project
        for j, mutator in enumerate(mutators):
            mutator_data = project_data[project_data['mutator'] == mutator]

            # Calculate bar position
            x_pos = i + (j * bar_width) - (len(mutators) * bar_width / 2) + (bar_width / 2)

            # If we have data for this mutator in this project
            if not mutator_data.empty:
                value = mutator_data['total'].values[0]

                # Plot the bar with consistent color
                bar = ax.bar(x_pos, value, width=bar_width, 
                       color=mutator_colors[mutator],
                       label=mutator_legend_labels[j] if i == 0 else "")

                # Add value on top of the bar
                ax.text(x_pos, value + 0.1, str(int(value)), 
                        ha='center', va='bottom', fontsize=9)
            else:
                # Plot an empty/zero bar to maintain spacing
                ax.bar(x_pos, 0, width=bar_width, color=mutator_colors[mutator],
                      label=mutator_legend_labels[j] if i == 0 else "")

    # Set x-axis labels and ticks
    ax.set_xlabel('Project')
    ax.set_ylabel('Number of Mutations')
    ax.set_title('Mutations by Project and Mutator')
    ax.set_xticks(range(len(project_labels)))
    ax.set_xticklabels(project_labels)
    ax.tick_params(axis='x', which='major', pad=15)

    # Add mutator indices below the x-axis
    for i, project_label in enumerate(project_labels):
        for j, mutator in enumerate(mutators):
            x_pos = i + (j * bar_width) - (len(mutators) * bar_width / 2) + (bar_width / 2)
            # Add index at a fixed position below the x-axis
            ax.text(x_pos, -50, mutator_indices[mutator], 
                    ha='center', va='top', fontsize=9, fontweight='bold')

    # Create legend with unique entries and position it outside the plot
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles, labels, 
               title='Mutator',
               loc='center left', 
               bbox_to_anchor=(1.0, 0.5))

    # Add a bit of padding to the top and bottom to accommodate the numbers and indices
    y_max = data['total'].max() if not data.empty else 10
    plt.ylim(-10, y_max * 1.1)  # Fixed bottom margin for indices

    # Adjust layout to make room for the legend
    plt.tight_layout()
    plt.subplots_adjust(right=0.85, bottom=0.2)  # Adjust margins for legend and indices

    return plt

# Create the plot
plot = create_mutator_bar_chart(df)
plot.show()

## Un-/Covered Mutants per Project + Variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT project_id, project_name, variant, total, covered, uncovered, covered_pct, uncovered_pct
FROM mv_mutation_results_by_project_variant
WHERE variant != 'ORIGINAL'
""", conn)

df = df.rename(columns={
  'covered_pct': 'Covered (%)',
  'uncovered_pct': 'Uncovered (%)',
})

df

## Percentage of Survived / Detected / ... Mutants per Project + Variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT
    project_id, project_name, variant, 
    survived_of_covered_pct, killed_of_covered_pct, timed_out_of_covered_pct, memory_error_of_covered_pct, run_error_of_covered_pct,
    survived_of_covered_pct_diff, killed_of_covered_pct_diff, timed_out_of_covered_pct_diff, memory_error_of_covered_pct_diff, run_error_of_covered_pct_diff
FROM mv_mutation_results_by_project_variant
WHERE variant NOT IN ('ORIGINAL', 'BASELINE')
""", conn)

df = df.rename(columns={
    'survived_of_covered_pct': 'Survived (%)',
    'killed_of_covered_pct': 'Killed (%)',
    'timed_out_of_covered_pct': 'Timed-Out (%)',
    'memory_error_of_covered_pct': 'Memory Error (%)',
    'run_error_of_covered_pct': 'Run Error (%)',
    'survived_of_covered_pct_diff': 'Survived (%) Diff.',
    'killed_of_covered_pct_diff': 'Killed (%) Diff.',
    'timed_out_of_covered_pct_diff': 'Timed-Out (%) Diff.',
    'memory_error_of_covered_pct_diff': 'Memory Error (%) Diff.',
    'run_error_of_covered_pct_diff': 'Run Error (%) Diff.',
})

df

## Percentage of Survived / Detected / ... Mutants per Project + Variant + Mutator

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT
    project_id, project_name, variant, mutator,
    survived_of_covered_pct, killed_of_covered_pct, timed_out_of_covered_pct, memory_error_of_covered_pct, run_error_of_covered_pct,
    survived_of_covered_pct_diff, killed_of_covered_pct_diff, timed_out_of_covered_pct_diff, memory_error_of_covered_pct_diff, run_error_of_covered_pct_diff
FROM mv_mutation_results_by_project_variant_mutator
WHERE variant IN ('IMPROVED_200_TRIES')
""", conn)

df = df.rename(columns={
    'survived_of_covered_pct': 'Survived (%)',
    'killed_of_covered_pct': 'Killed (%)',
    'timed_out_of_covered_pct': 'Timed-Out (%)',
    'memory_error_of_covered_pct': 'Memory Error (%)',
    'run_error_of_covered_pct': 'Run Error (%)',
    'survived_of_covered_pct_diff': 'Survived (%) Diff.',
    'killed_of_covered_pct_diff': 'Killed (%) Diff.',
    'timed_out_of_covered_pct_diff': 'Timed-Out (%) Diff.',
    'memory_error_of_covered_pct_diff': 'Memory Error (%) Diff.',
    'run_error_of_covered_pct_diff': 'Run Error (%) Diff.',
})

df

## Overview of Mutation Scores and Changes per Project + Variant

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

def fetch_mutation_data(conn, variants=None):
    """Fetch and prepare mutation data from database."""
    query = "SELECT * FROM mv_mutation_results_by_project_variant_mutator"
    if variants:
        variant_list = "', '".join(variants)
        query += f" WHERE variant IN ('{variant_list}')"

    results = pd.read_sql_query(query, conn)

    # Rename columns to match visualization expectations
    return results.rename(columns={
        'detected_pct': 'detection_rate',
        'detected_pct_diff': 'improvement'
    })

def get_sorted_variants(mutator_results, conn):
    """Get variants sorted by their order from database."""
    all_variants = sorted(
        mutator_results['variant'].unique(), 
        key=lambda v: pd.read_sql_query(f"SELECT variant_order('{v}')", conn).iloc[0, 0]
    )
    improvement_variants = [v for v in all_variants if v not in ['ORIGINAL', 'INITIAL', 'BASELINE']]
    return all_variants, improvement_variants

def create_variant_color_mapping(variants):
    """Create a consistent color mapping for variants using seaborn's colorblind palette."""
    # Use seaborn's colorblind-friendly palette
    color_palette = sns.color_palette("colorblind", n_colors=len(variants))

    # Convert RGB tuples to hex codes using matplotlib's color conversion
    hex_palette = [mcolors.to_hex(color) for color in color_palette]

    return {variant: hex_palette[i] for i, variant in enumerate(variants)}

def calculate_y_axis_limits(mutator_results, project_ids):
    """Calculate consistent y-axis limits across all plots."""
    detection_y_max = 0
    improvement_y_max = 0
    improvement_y_min = 0

    for project_id in project_ids:
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        detection_y_max = max(detection_y_max, project_data['detection_rate'].max() * 1.1)

        project_improvements = project_data[project_data['improvement'].notna()]['improvement']
        if not project_improvements.empty:
            improvement_y_max = max(improvement_y_max, project_improvements.max() * 1.1)
            improvement_y_min = min(improvement_y_min, project_improvements.min() * 1.1)

    return detection_y_max, improvement_y_max, improvement_y_min

def plot_detection_rates(ax, project_data, all_variants, all_mutators, variant_colors):
    """Plot detection rate bars for a project."""
    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(all_variants)
    total_width = width * len(all_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(all_variants):
        variant_data = project_data[project_data['variant'] == variant]
        detection_rates = {row['mutator']: row['detection_rate'] for _, row in variant_data.iterrows()}

        for mutator, pos in mutator_positions.items():
            rate = detection_rates.get(mutator, 0)
            if rate > 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, rate, width, color=variant_colors[variant])

def plot_improvements(ax, project_data, improvement_variants, all_mutators, variant_colors):
    """Plot improvement bars for a project."""
    if not improvement_variants:
        return

    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(improvement_variants)
    total_width = width * len(improvement_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(improvement_variants):
        variant_data = project_data[project_data['variant'] == variant]
        improvements = {row['mutator']: row['improvement'] 
                       for _, row in variant_data.iterrows() 
                       if row['improvement'] is not None}

        for mutator, pos in mutator_positions.items():
            impr = improvements.get(mutator, 0)
            if impr != 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, impr, width, color=variant_colors[variant])

def configure_axis(ax, title, ylabel, x_min, x_max, y_min, y_max, all_mutators, show_xticklabels=False):
    """Configure axis properties."""
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(np.arange(len(all_mutators)))

    if show_xticklabels:
        ax.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax.set_xticklabels([])

    ax.grid(axis='y', linestyle='--', alpha=0.7)

# Main visualization function
def visualize_mutation_results(conn, variants=None):
    """Create comprehensive visualization of mutation testing results."""
    # Set seaborn style for better aesthetics
    sns.set_style("whitegrid")

    # Fetch and prepare data
    mutator_results = fetch_mutation_data(conn, variants)
    all_variants, improvement_variants = get_sorted_variants(mutator_results, conn)
    all_mutators = sorted(mutator_results['mutator'].unique())
    project_ids = mutator_results['project_id'].unique()
    project_count = len(project_ids)

    # Calculate axis limits
    detection_y_max, improvement_y_max, improvement_y_min = calculate_y_axis_limits(
        mutator_results, project_ids)

    # Create color mapping using seaborn's colorblind palette
    variant_colors = create_variant_color_mapping(all_variants)

    # Create figure and grid
    fig = plt.figure(figsize=(18, 5 + 4 * project_count))
    gs = GridSpec(project_count, 2, width_ratios=[3, 2])

    # Set fixed x-axis limits
    x_min, x_max = -0.5, len(all_mutators) - 0.5

    # Create legend
    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=variant_colors[variant]) 
                     for variant in all_variants]
    fig.legend(
        legend_handles, all_variants, loc='upper center',
        ncol=min(len(all_variants), 5), bbox_to_anchor=(0.5, 0.98), fontsize='small'
    )

    # Create plots for each project
    for i, project_id in enumerate(project_ids):
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        if project_data.empty:
            continue

        # Get project name
        project_name = pd.read_sql_query(
            f"SELECT project_name({project_id})", conn).iloc[0, 0]

        # Create subplots
        ax1 = fig.add_subplot(gs[i, 0])  # Detection rate plot
        ax2 = fig.add_subplot(gs[i, 1])  # Improvement plot

        # Plot detection rates
        plot_detection_rates(ax1, project_data, all_variants, all_mutators, variant_colors)
        configure_axis(
            ax1, f'Detection Rate - Project ID: {project_id} - {project_name}',
            'Detection Rate (%)', x_min, x_max, 0, detection_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )

        # Plot improvements
        plot_improvements(ax2, project_data, improvement_variants, all_mutators, variant_colors)
        configure_axis(
            ax2, f'Improvement - Project ID: {project_id} - {project_name}',
            'Improvement (%)', x_min, x_max, improvement_y_min, improvement_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )
        ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)

    # Adjust layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    plt.subplots_adjust(hspace=0.3, top=0.88)

    return fig

variants_to_plot = ['INITIAL', 'NAIVE_200_TRIES', 'IMPROVED_200_TRIES']
fig = visualize_mutation_results(conn, variants_to_plot)
plt.show()


## Number of newly killed mutants per project + variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT *
FROM mv_generalization_effects
""", conn)

df[['project_id', 'project_name', 'a_variant', 'b_variant', 'killed_mutations']]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Assuming df is your DataFrame from the SQL query

# Filter data for the two plots
original_df = df[df['a_variant'] == 'ORIGINAL']
initial_df = df[df['a_variant'] == 'INITIAL']

# Function to create the grouped bar chart with consistent colors
def create_grouped_bar_chart(data, title):
    # Sort by project_id and b_variant_order
    data = data.sort_values(['project_id', 'b_variant_order'])

    plt.figure(figsize=(16, 8))

    # Create combined project identifier (id + name)
    data['project_label'] = data['project_id'].astype(str) + ': ' + data['project_name']

    # Get unique project labels and variants
    project_labels = data['project_label'].unique()
    variants = data.sort_values('b_variant_order')['b_variant'].unique()

    # Create a color map for variants (using the new approach)
    color_map = plt.colormaps['tab10']
    variant_colors = {variant: color_map(i % 10) for i, variant in enumerate(variants)}

    # Set up the plot
    ax = plt.subplot(111)
    bar_width = 0.8 / len(variants)

    # For each project, plot bars for each variant
    for i, project_label in enumerate(project_labels):
        project_data = data[data['project_label'] == project_label]

        # For each variant, find its data for this project
        for j, variant in enumerate(variants):
            variant_data = project_data[project_data['b_variant'] == variant]

            # Calculate bar position
            x_pos = i + (j * bar_width) - (len(variants) * bar_width / 2) + (bar_width / 2)

            # If we have data for this variant in this project
            if not variant_data.empty:
                value = variant_data['killed_mutations'].values[0]

                # Plot the bar with consistent color
                bar = ax.bar(x_pos, value, width=bar_width, 
                       color=variant_colors[variant],
                       label=variant if i == 0 else "")

                # Add text on top of the bar
                ax.text(x_pos, value + 0.1, str(int(value)), 
                        ha='center', va='bottom', fontsize=9)
            else:
                # Plot an empty/zero bar to maintain spacing
                ax.bar(x_pos, 0, width=bar_width, color=variant_colors[variant],
                      label=variant if i == 0 else "")

    # Set x-axis labels and ticks
    ax.set_xlabel('Project')
    ax.set_ylabel('Newly Killed Mutations')
    ax.set_title(title)
    ax.set_xticks(range(len(project_labels)))
    ax.set_xticklabels(project_labels, rotation=45, ha='right')

    # Create legend with unique entries
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), title='Variant')

    # Add a bit of padding to the top to accommodate the numbers
    y_max = data['killed_mutations'].max() if not data.empty else 10
    plt.ylim(0, y_max * 1.1)

    plt.tight_layout()
    return plt

# Create the two plots
plot1 = create_grouped_bar_chart(original_df, 'Newly Killed Mutations by Project (a_variant = ORIGINAL)')
plot1.show()

plot2 = create_grouped_bar_chart(initial_df, 'Newly Killed Mutations by Project (a_variant = INITIAL)')
plot2.show()


## Effects of Generalization on Test Suite Size + Runtime

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT *
FROM mv_generalization_effects
WHERE a_variant = 'ORIGINAL' AND b_variant = 'IMPROVED_200_TRIES'
""", conn)

In [ ]:
df[['project_name', 'a_variant', 'b_variant', 'tests_before', 'added_tests', 'removed_tests', 'tests_after', 'tests_delta', 'tests_delta_pct']]

In [ ]:
df[['project_name', 'a_variant', 'b_variant', 'lines_before', 'added_lines', 'removed_lines', 'lines_after', 'lines_delta', 'lines_delta_pct']]

In [ ]:
df[['project_name', 'a_variant', 'b_variant', 'runtime_before', 'added_runtime', 'removed_runtime', 'runtime_after', 'runtime_delta', 'runtime_delta_pct']]

## Description of Generalizations That Killed New Mutants

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT *
FROM mv_mutation_status_changes
WHERE a_is_detected IS FALSE AND b_status = 'KILLED'
""", conn)

df

## Comparison of Detected vs. Undetected Mutants

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT * FROM mv_mutation_detection_comparison
""", conn)

df